In [ ]:
import sys
import langchain
import chromadb
import openai
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma



In [ ]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
# Load raw documents from data/1_source/
loader = DirectoryLoader(
    "../data/1_source/",
    glob="**/*.txt",
    loader_cls=TextLoader
)

docs = loader.load()
print(f"Loaded {len(docs)} documents")

In [ ]:
# Inspect a document
print(docs[0].page_content[:500])   # first 500 chars
print("---")
print(docs[0].metadata)             # source path, etc.

In [ ]:
# Chunk the documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(docs)
print(f"Total chunks: {len(chunks)}")
print(f"Avg chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")

In [ ]:
# Inspect a few chunks to sanity check
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content)

In [ ]:
# Generate embeddings + store in ChromaDB
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # cheapest + good

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="../chroma_db/"
)

print(f"Stored {vectorstore._collection.count()} chunks in ChromaDB ✅")